In [36]:
# To standardize all the classes, we need to use the abstrct classes to implement it. 
from abc import ABC, abstractmethod

In [37]:
class Runnable(ABC): 
    
    @abstractmethod
    def invoke(input_data): 
        pass 

In [38]:
# here we create the copy or trying to making the class to get better idea how chains internally depend on runnable class. 

import random

class NakliLLM(Runnable): 

    def __init__(self):
        print('LLM created successfully.')

    def invoke(self, prompt): 
        response_list = [
            'AI is revolutionary in the upcoming years', 
            'IPL is a biggest league in the world', 
            'Delhi is the capital of India', 
            'World is under going through the crisis'
        ]
        return {'response' : random.choice(response_list)} 


    # we dont remove the predict method, but we just return the warning message that this method will depricated soon. 
    def predict(self, prompt): 

        response_list = [
            'AI is revolutionary in the upcoming years', 
            'IPL is a biggest league in the world', 
            'Delhi is the capital of India', 
            'World is under going through the crisis'
        ]
        return {'response' : random.choice(response_list)}

In [39]:
# here we also create one more class for the same purpose. 
class NakliPromptTemplate(Runnable): 

    def __init__(self, template, input_variables): 
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict): 
        return self.template.format(**input_dict)

    def format(self, input_dict): 
        return self.template.format(**input_dict)

In [40]:
# create new class that is responible for communication between two components or runnable. 
class RunnableConnector(Runnable): 
    def __init__(self, runnable_list): 
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        
        for runnable in self.runnable_list: 
            input_data = runnable.invoke(input_data)

        return input_data
    

In [41]:
# create new class that is used for the String Parser where we try to replicate this class. 
class NakliStrOutputParser(Runnable): 

    def __init__(self): 
        print('Parser is created successfully.') 

    def invoke(self, input_data): 
        return input_data['response']

In [42]:
# create the template. 
template = NakliPromptTemplate(
    template = 'Write the {length} poem about the {topic}', 
    input_variables = ['length', 'topic']
)

# create the llm. 
llm = NakliLLM()

# create the string parser. 
parser = NakliStrOutputParser()

LLM created successfully.
Parser is created successfully.


In [43]:
chain = RunnableConnector([template, llm, parser])

In [44]:
chain.invoke({
    'length' : 'long', 
    'topic' : 'English Language'
})

'AI is revolutionary in the upcoming years'

### Here we build a setup where two chains are communicate with each other.

In [45]:
template1 = NakliPromptTemplate(
    template = 'Generate the funny joke about the {topic}', 
    input_variables = ['topic']
)

template2 = NakliPromptTemplate(
    template = 'Explain about the meaning of the given joke, and the joke is {response}', 
    input_variables = ['response']
)

In [46]:
llm = NakliLLM()

LLM created successfully.


In [47]:
parser = NakliStrOutputParser()

Parser is created successfully.


In [57]:
chain1 = RunnableConnector([template1, llm])

In [61]:
response = chain2.invoke({
    'response' : 'Here is the joke'
})
response

'Delhi is the capital of India'

In [62]:
chain2 = RunnableConnector([template2, llm, parser])

In [63]:
result = chain2.invoke({
    'response' : 'Here is the joke'
})
result

'Delhi is the capital of India'

In [65]:
final_chain = RunnableConnector([chain1, chain2])

In [66]:
final_chain.invoke({
    'topic' : 'Cricket'
})

'IPL is a biggest league in the world'